# Installing necessary libraries

In [ ]:
!pip install skimpy
!pip install shap

# Import necessary libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from lime.lime_tabular import LimeTabularExplainer
from skimpy import skim
import shap

# Load and Glimpse of Dataset

In [ ]:
# Load dataset
file_path = '/kaggle/input/cleaned-preprocessed-700-csv/cleaned_preprocessed_700.csv'
cleaned_preprocessed_700 = pd.read_csv(file_path)

# Quick summary of the dataset using skimpy
skim(cleaned_preprocessed_700)

# A bit of EDA

In [ ]:
# EDA: Check for missing values
print("\nMissing values:\n", cleaned_preprocessed_700.isnull().sum())

# EDA: Summary statistics for numerical features
print("\nSummary statistics:\n", cleaned_preprocessed_700.describe())

# EDA: Distribution of numerical features
plt.figure(figsize=(15, 10))
cleaned_preprocessed_700.hist(bins=30, color='lightgreen', edgecolor='black')
plt.tight_layout()
plt.show()

# EDA: Count plot for categorical features
categorical_columns = ['Fertility', 'Photoperiod', 'Category_pH', 'Soil_Type', 'Season']
plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_columns, 1):
    plt.subplot(2, 3, i)
    sns.countplot(x=col, data=cleaned_preprocessed_700, palette='Greens')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Data Preprocessing

In [ ]:
# Data preprocessing
label_encoders = {col: LabelEncoder() for col in categorical_columns}

for col in categorical_columns:
    cleaned_preprocessed_700[col] = label_encoders[col].fit_transform(cleaned_preprocessed_700[col])

# Encode the target variable (crop names)
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(cleaned_preprocessed_700['Name'])

# Define features and drop the target column
X = cleaned_preprocessed_700.drop(columns=['Name'])

# Standardize numerical columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Correlation Analysis
# Select only numerical columns for correlation
numerical_columns = cleaned_preprocessed_700.select_dtypes(include=[np.number]).columns
correlation_matrix = cleaned_preprocessed_700[numerical_columns].corr()

# Plotting the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='Greens', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

# Model Training and Evaluation

In [ ]:
# Define models
models = {
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': XGBClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=200)
}

# Train and evaluate models
model_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    model_results[name] = model.score(X_test, y_test)

# Print model performance
print("Model Performance:")
for model_name, accuracy in model_results.items():
    print(f"{model_name}: {accuracy:.4f}")

# LIME Explanation for a Single Instance

In [ ]:
# Initialize LIME explainer for Random Forest (as an example)
lime_explainer = LimeTabularExplainer(
    X_train,
    feature_names=cleaned_preprocessed_700.drop(columns=['Name']).columns,
    class_names=target_encoder.classes_,
    mode='classification'
)

# Explaining a single instance
instance_idx = 0
exp = lime_explainer.explain_instance(X_test[instance_idx], models['Random Forest'].predict_proba, num_features=5)
exp.show_in_notebook()

# SHAP Explanation for Global Feature Importance

In [ ]:
# Compute SHAP values for RF
# Use a subset of the test data for efficiency
X_test_subset = X_test[:100]

# Create TreeExplainer
shap_explainer = shap.TreeExplainer(models['Random Forest'])

# Compute SHAP values
shap_values = shap_explainer.shap_values(X_test_subset)

# Plot feature importance for each crop (class)
for i, class_name in enumerate(target_encoder.classes_):
    print(f"\nFeature importance for crop: {class_name}")

    # Use the summary_plot to show SHAP values
    shap.summary_plot(shap_values[i], X_test_subset, feature_names=X.columns, show=False)

    # Adjust the title to reflect the class name
    plt.title(f"SHAP Summary Plot for {class_name}")

    # Customize color of the plot (as a workaround)
    plt.gca().collections[0].set_array(np.array(shap_values[i]).flatten())
    plt.gca().collections[0].set_cmap('Greens')  # Use a green colormap
    plt.show()

#  Bar Plot for Feature Importance for Clearer Understanding

In [ ]:
# Compute SHAP values for RF
# Use a subset of the test data for efficiency
X_test_subset = X_test[:100]

# Create TreeExplainer
shap_explainer = shap.TreeExplainer(models['Random Forest'])

# Compute SHAP values
shap_values = shap_explainer.shap_values(X_test_subset)

# Check the shape of shap_values
print("Shape of shap_values:", np.shape(shap_values))

# Prepare an empty list to store mean SHAP values for each crop
mean_shap_values_per_crop = []

# Compute mean absolute SHAP values for each crop
if isinstance(shap_values, list):  # For the case of multiple classes
    for i, class_shap_values in enumerate(shap_values):
        mean_shap = np.mean(np.abs(class_shap_values), axis=0)
        mean_shap_values_per_crop.append((target_encoder.classes_[i], mean_shap))
else:
    mean_shap = np.abs(shap_values).mean(axis=0)
    mean_shap_values_per_crop.append(("Overall", mean_shap))

# Plotting feature importance for each crop
for crop, mean_shap in mean_shap_values_per_crop:
    feature_importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Mean SHAP Value': mean_shap
    })

    # Sort the DataFrame by Mean SHAP Value
    feature_importance_df = feature_importance_df.sort_values(by='Mean SHAP Value', ascending=False)

    # Plotting the feature importance using a bar plot
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df['Feature'], feature_importance_df['Mean SHAP Value'], color='lightgreen')
    plt.xlabel('Mean Absolute SHAP Value', color='green')
    plt.ylabel('Features', color='green')
    plt.title(f'Feature Importance based on Mean SHAP Values for {crop}', color='green')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

#  Gradient Boosting Global Feature Importance Plot

In [ ]:
#visualize feature importances (Gradient Boosting)
importances = models['Gradient Boosting'].feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Global Feature Importances (Gradient Boosting)", color='green')
plt.bar(range(X_train.shape[1]), importances[indices], align="center", color='lightgreen')
plt.xticks(range(X_train.shape[1]), X.columns[indices], rotation=90, color='black')
plt.xlabel("Features", color='green')
plt.ylabel("Importance", color='green')
plt.tight_layout()
plt.show()
